# Microsoft Agent Framework

In [ ]:
import os
from random import randint
from typing import Annotated

from dotenv import load_dotenv
from pydantic import Field

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient

# Load environment variables from the .env file
load_dotenv()

# Verify the variables loaded
print("Endpoint loaded:", bool(os.getenv("AZURE_OPENAI_ENDPOINT")))
print("API key loaded:", bool(os.getenv("AZURE_OPENAI_API_KEY")))
print("Deployment loaded:", bool(os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")))

## Set logging

In [ ]:
# pip install azure-monitor-opentelemetry
# Send traces to Azure Application Insights (viewable in the Azure / Foundry portal).
#
# Get the connection string from:
#   Azure portal -> your Application Insights resource -> Overview -> "Connection String"
#   (this is the App Insights connected to your Foundry project's Tracing tab)
# Then add it to your .env file as:
#   APPLICATIONINSIGHTS_CONNECTION_STRING=InstrumentationKey=...;IngestionEndpoint=https://...
#
# Using the connection string directly avoids AAD auth, so it works even when the
# Foundry resource is in a different tenant than your `az login`.
import os

from azure.monitor.opentelemetry import configure_azure_monitor
from agent_framework.observability import enable_instrumentation

conn = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
if not conn:
    raise ValueError(
        "APPLICATIONINSIGHTS_CONNECTION_STRING is not set. "
        "Add it to your .env (copy from the App Insights resource -> Connection String)."
    )

configure_azure_monitor(connection_string=conn)
enable_instrumentation(enable_sensitive_data=True)  # include prompts / thoughts / tool args+results
print("Azure Monitor tracing configured. Traces appear in the portal in ~30-90s.")


## Normal response

In [ ]:
client = OpenAIChatClient(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
)

agent = Agent(
    client=client,
    name="HelloAgent",
    instructions="You are a friendly assistant. Keep your answers brief.",
)

response = await agent.run("Say hello and introduce yourself in one sentence.")
print(response)

## Streaming

In [ ]:
# Streaming: receive tokens as they are generated
print("Agent (streaming): ", end="", flush=True)
async for chunk in agent.run("Tell me a one-sentence fun fact.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

## Tool calling

### Define tool

In [ ]:
# NOTE: approval_mode="never_require" is for sample brevity.
# Use "always_require" in production for user confirmation before tool execution.
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."

### Agent with tool

In [ ]:
agent = Agent(
    client=client,
    name="WeatherAgent",
    instructions="You are a helpful weather agent. Use the get_weather tool to answer questions.",
    tools=[get_weather],
    
)

In [ ]:
response = await agent.run("What's the weather like in Paris today?")
print(response)

## Car price agent (parallel lookups + calculator tool)

A single agent with two tools merged together:

- **`get_car_price`** — looks up one brand's price. The model calls it once per brand, and
  Agent Framework runs those calls in parallel (see the interleaved `START`/`END` prints).
- **`add`** — a deterministic calculator tool. The agent is told to never do math itself and
  must call `add` for any total, so the arithmetic is always correct.

Flow: `get_car_price` (parallel per brand) → `add` → final answer. In the trace this shows
as two `execute_tool get_car_price` spans plus one `execute_tool add`, all under a single
`invoke_agent CarPriceAgent`.


In [ ]:
import asyncio
import time
from datetime import datetime

# Static price list (USD) for the demo.
CAR_PRICES = {"honda": 28000, "toyota": 31000}


@tool(approval_mode="never_require")
async def get_car_price(
    brand: Annotated[str, Field(description="Car brand, e.g. 'honda' or 'toyota'.")],
) -> str:
    """Get the price of a single car for the given brand."""
    print(f"[{datetime.now():%H:%M:%S.%f}] get_car_price START (brand={brand})")
    await asyncio.sleep(2)  # simulate a slow lookup so parallelism is observable
    price = CAR_PRICES.get(brand.strip().lower())
    result = f"{brand}: ${price}" if price is not None else f"{brand}: unknown brand"
    print(f"[{datetime.now():%H:%M:%S.%f}] get_car_price END   (brand={brand}) -> {result}")
    return result


@tool(approval_mode="never_require")
def add(
    numbers: Annotated[list[float], Field(description="The list of numbers to add together.")],
) -> str:
    """Add a list of numbers and return their total."""
    print(f"[{datetime.now():%H:%M:%S.%f}] add() called with {numbers}")
    return f"The sum of {numbers} is {sum(numbers)}."


# One agent, both tools: parallel price lookups + deterministic addition.
car_agent = Agent(
    client=client,
    name="CarPriceAgent_Parallel",
    instructions=(
        "You are a car pricing assistant. Use get_car_price to look up each brand's price, "
        "requesting multiple brands together so the lookups run in parallel. "
        "For ANY addition or total, do NOT calculate it yourself — call the add tool."
    ),
    tools=[get_car_price, add],
)

start = time.perf_counter()
response = await car_agent.run(
    "What is the total price of 1 car from Honda brand and 2 cars from Toyota brand?"
)
print(f"\nWall-clock time: {time.perf_counter() - start:.1f}s\n")
print(response)
